# **1. SQLite**
SQLite는 별도의 데이터베이스 서버를 설치하지 않고 파일로 사용하는 데이터베이스


In [1]:
import sqlite3
from pathlib import Path

In [2]:
DATA_DIR = Path("data")
DATA_DIR.mkdir(exist_ok=True)
simple_path = DATA_DIR / "hello.db"

In [3]:
conn_simple = sqlite3.connect(simple_path)
print("연결 성공", simple_path.resolve())

연결 성공 C:\LANG_CHAIN_2026\2026-05-19_KDT_lang_chain\01_class\2026_10_30_agent\data\hello.db


In [4]:
conn_simple.execute("""
create table students(
    id integer primary key,
    name varchar(20) not null,
    score integer not null
)
""")

In [5]:
students = [(1, "김사과", 90), (2, "반하나", 85), (3, "오렌지", 95)]
conn_simple.executemany("insert into students (id, name, score) values (?, ?, ?)", students)
conn_simple.commit()
print("학생 데이터 저장 완료")

학생 데이터 저장 완료


In [6]:
rows = conn_simple.execute("select name, score from students where score >= ? order by score desc", (90,)).fetchall()

for name, score in rows:
    print(name, score)

오렌지 95
김사과 90


In [7]:
import pandas as pd

In [8]:
pd.read_sql_query("SELECT * FROM students ORDER BY score DESC", conn_simple)

,id,name,score
0,3,오렌지,95
1,1,김사과,90
2,2,반하나,85


In [9]:
conn_simple.close()
print("연습용 DB 연결 종료")

연습용 DB 연결 종료


In [10]:
import requests

In [11]:
path = DATA_DIR / "Chinook.db"
if not path.exists():
    url = "https://storage.googleapis.com/benchmarks-artifacts/chinook/Chinook.db"
    response = requests.get(url, timeout=30)
    response.raise_for_status()
    path.write_bytes(response.content)
    
print("DB 파일: ", path.resolve())
conn = sqlite3.connect(path)
print("연결 성공")

DB 파일:  C:\LANG_CHAIN_2026\2026-05-19_KDT_lang_chain\01_class\2026_10_30_agent\data\Chinook.db
연결 성공


In [12]:
tables = conn.execute("select name from sqlite_master where type='table' order by name").fetchall()
print([name for (name,) in tables])

['Album', 'Artist', 'Customer', 'Employee', 'Genre', 'Invoice', 'InvoiceLine', 'MediaType', 'Playlist', 'PlaylistTrack', 'Track']


In [13]:
pd.read_sql_query("select * from customer limit 5", conn)

,CustomerId,FirstName,LastName,Company,Address,City,State,Country,PostalCode,Phone,Fax,Email,SupportRepId
0,1,Luís,Gonçalves,Embraer - Empresa Brasileira de Aeronáutica S.A.,"Av. Brigadeiro Faria Lima, 2170",São José dos Campos,SP,Brazil,12227-000,+55 (12) 3923-5555,+55 (12) 3923-5566,luisg@embraer.com.br,3
1,2,Leonie,Köhler,NaN,Theodor-Heuss-Straße 34,Stuttgart,NaN,Germany,70174,+49 0711 2842222,NaN,leonekohler@surfeu.de,5
2,3,François,Tremblay,NaN,1498 rue Bélanger,Montréal,QC,Canada,H2G 1A7,+1 (514) 721-4711,NaN,ftremblay@gmail.com,3
3,4,Bjørn,Hansen,NaN,Ullevålsveien 14,Oslo,NaN,Norway,0171,+47 22 44 22 22,NaN,bjorn.hansen@yahoo.no,4
4,5,František,Wichterlová,JetBrains s.r.o.,Klanova 9/506,Prague,NaN,Czech Republic,14700,+420 2 4172 5555,+420 2 4172 5555,frantisekw@jetbrains.com,4


In [14]:
# 주문별 고객 이름과 주문 금액(customer, invoice)
pd.read_sql_query("""
    select c.firstname, c.lastname, i.total from customer as c join invoice as i on c.customerid = i.customerid limit 10
""", conn)

,FirstName,LastName,Total
0,Leonie,Köhler,1.98
1,Bjørn,Hansen,3.96
2,Daan,Peeters,5.94
3,Mark,Philips,8.91
4,John,Gordon,13.86
5,Fynn,Zimmermann,0.99
6,Niklas,Schröder,1.98
7,Dominique,Lefebvre,1.98
8,Wyatt,Girard,3.96
9,Hugh,O'Reilly,5.94


In [15]:
# 국가별 주문 총액 상위 5개
pd.read_sql_query("""
    select billingcountry, round(sum(total), 2) as revenue from invoice group by billingcountry order by revenue desc limit 5
""", conn)

,BillingCountry,revenue
0,USA,523.06
1,Canada,303.96
2,France,195.10
3,Brazil,190.10
4,Germany,156.48


# **2. LangChain으로 연결**

In [16]:
from langchain_community.utilities import SQLDatabase

C:\Users\hurwa\AppData\Local\Temp\ipykernel_905232\1266060898.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.utilities import SQLDatabase


In [17]:
# as_posix(): 경로 구분자를 / 형태로 변환
db = SQLDatabase.from_uri(f"sqlite:///{path.resolve().as_posix()}")
print("DB 종류: ", db.dialect)
print("테이블: ",db.get_usable_table_names())
print("\nInvoice 테이블 구조 일부: ")
print(db.get_table_info(['Invoice'])[:1200])

DB 종류:  sqlite
테이블:  ['Album', 'Artist', 'Customer', 'Employee', 'Genre', 'Invoice', 'InvoiceLine', 'MediaType', 'Playlist', 'PlaylistTrack', 'Track']

Invoice 테이블 구조 일부: 

CREATE TABLE "Invoice" (
	"InvoiceId" INTEGER NOT NULL, 
	"CustomerId" INTEGER NOT NULL, 
	"InvoiceDate" DATETIME NOT NULL, 
	"BillingAddress" NVARCHAR(70), 
	"BillingCity" NVARCHAR(40), 
	"BillingState" NVARCHAR(40), 
	"BillingCountry" NVARCHAR(40), 
	"BillingPostalCode" NVARCHAR(10), 
	"Total" NUMERIC(10, 2) NOT NULL, 
	PRIMARY KEY ("InvoiceId"), 
	FOREIGN KEY("CustomerId") REFERENCES "Customer" ("CustomerId")
)

/*
3 rows from Invoice table:
InvoiceId	CustomerId	InvoiceDate	BillingAddress	BillingCity	BillingState	BillingCountry	BillingPostalCode	Total
1	2	2009-01-01 00:00:00	Theodor-Heuss-Straße 34	Stuttgart	None	Germany	70174	1.98
2	4	2009-01-02 00:00:00	Ullevålsveien 14	Oslo	None	Norway	0171	3.96
3	8	2009-01-03 00:00:00	Grétrystraat 63	Brussels	None	Belgium	1000	5.94
*/


In [18]:
import os
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI

MODEL = os.getenv("OPENAI_MODEL", "gpt-5.6-luna")
llm = ChatOpenAI(model=MODEL, temperature=0)